<a href="https://colab.research.google.com/github/maliksonmez/arnavutkoy-air-quality-forecast/blob/main/notebooks/03_external_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Dış veri: meteoroloji + Copernicus CAMS

Ödev 1 geri bildirimi üzerine eklendi: model artık yalnızca istasyonun kendi geçmişini görmeyecek.

**Bu notebook ne yapar?**
1. İstasyon verisini GitHub'dan okur (EDA ile aynı temizlik kuralları).
2. Open-Meteo'dan istasyon noktası için **saatlik meteoroloji** (ERA5 tabanlı arşiv) çeker.
3. Open-Meteo'dan **Copernicus CAMS hava kalitesi tahminlerini** (PM10, PM2.5, NO2, SO2, Sahra tozu) çeker.
4. Üçünü saat bazında birleştirir, günlük özet tabloyu üretir ve iki CSV dosyası olarak kaydeder.
5. Kapsam, CAMS–istasyon farkı ve saat dilimi uyumu için hızlı kontroller yapar.

Çalıştırma: *Çalışma zamanı → Tümünü çalıştır*. Dosya yüklemeye gerek yok. API anahtarı gerekmez.

Kaynak ve atıf: Open-Meteo (CC BY 4.0); CAMS European air quality forecasts (Copernicus Atmosphere Monitoring Service); ERA5 (Copernicus Climate Change Service).

In [ ]:
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# İstasyon: İstanbul - Arnavutköy (SİM, işletici İBB) — Öztürk Cd., Adnan Menderes Mah.
# Koordinat SİM portalındaki istasyon haritası Google Haritalar ile eşleştirilerek alındı (28 Ekim 2026).
ENLEM, BOYLAM = 41.220937, 28.707547

BASLANGIC = "2023-09-16"   # istasyon verisinin ilk günü
BITIS = "2026-09-16"       # istasyon verisinin son günü
SAAT_DILIMI = "Europe/Istanbul"   # Open-Meteo zamanları yerel saatle döndürsün

VERI_URL = (
    "https://github.com/maliksonmez/arnavutkoy-air-quality-forecast/"
    "raw/main/data/arnavutkoy_hourly_2023-09-16_2026-09-16.xlsx"
)

kirleticiler = ["PM10", "PM2_5", "SO2", "NO2"]
print("Hazır.")

## 1. İstasyon verisi (EDA ile aynı kurallar)

In [ ]:
ham_veri = pd.read_excel(VERI_URL, header=None)

veri = ham_veri.iloc[2:].copy()
veri.columns = ["tarih"] + kirleticiler
veri = veri.reset_index(drop=True)
veri["tarih"] = pd.to_datetime(veri["tarih"], errors="coerce")

for k in kirleticiler:
    metin = veri[k].astype("string").str.strip()
    veri[k] = pd.to_numeric(metin.str.replace(",", ".", regex=False), errors="coerce").astype(float)

veri = veri.sort_values("tarih").reset_index(drop=True)
assert veri["tarih"].notna().all() and not veri["tarih"].duplicated().any()

# SİM zaman damgaları tam saat değil (ör. 00:00:56). Dış verilerle eşleşmesi için
# saat başına indiriyoruz; ölçüm o saate aittir.
print("Tam saat olmayan zaman damgası:", (veri["tarih"] != veri["tarih"].dt.floor("h")).sum())
veri["tarih"] = veri["tarih"].dt.floor("h")
assert not veri["tarih"].duplicated().any()

# EDA kararı: tekrarlayan 985 kayıtları analiz kopyasında eksik sayılır.
for k in ["PM10", "PM2_5"]:
    veri.loc[veri[k].eq(985), k] = np.nan

istasyon = veri.set_index("tarih")
print("İstasyon:", istasyon.index.min(), "→", istasyon.index.max(), "|", len(istasyon), "saat")

## 2. Open-Meteo yardımcı fonksiyonu

Uzun tarih aralıkları parçalara bölünür; hata olursa API'nin verdiği sebep yazdırılır.

In [ ]:
def open_meteo_cek(url, parametreler, baslangic, bitis, parca_gun=None, deneme=3):
    """Open-Meteo'dan saatlik veri çeker; isteğe bağlı olarak tarih aralığını parçalara böler."""
    if parca_gun is None:
        araliklar = [(pd.Timestamp(baslangic), pd.Timestamp(bitis))]
    else:
        sinirlar = pd.date_range(baslangic, bitis, freq=f"{parca_gun}D")
        araliklar = [(s, min(s + pd.Timedelta(days=parca_gun - 1), pd.Timestamp(bitis))) for s in sinirlar]

    parcalar = []
    for bas, son in araliklar:
        p = dict(parametreler, start_date=bas.strftime("%Y-%m-%d"), end_date=son.strftime("%Y-%m-%d"))
        for i in range(deneme):
            yanit = requests.get(url, params=p, timeout=120)
            if yanit.status_code == 200:
                break
            if yanit.status_code == 400:   # yanlış parametre: tekrar denemenin anlamı yok
                raise ValueError(f"Open-Meteo hatası: {yanit.json().get('reason')}")
            time.sleep(5 * (i + 1))
        yanit.raise_for_status()
        saatlik = yanit.json()["hourly"]
        parcalar.append(pd.DataFrame(saatlik))
        time.sleep(0.5)   # ücretsiz servise nazik davranalım

    tablo = pd.concat(parcalar, ignore_index=True)
    tablo["time"] = pd.to_datetime(tablo["time"])
    return tablo.drop_duplicates("time").set_index("time").sort_index()

## 3. Meteoroloji (Open-Meteo Historical Weather API, ERA5 tabanlı)

Bir hafta öncesinden başlıyoruz ki ilk günlerin gecikmeli (lag) girdileri de dolu olsun.

> **Sızıntı notu:** Bu değerler "gerçekleşen" hava durumudur. Tahmin modelinde yalnızca **bugünün ve geçmiş günlerin** meteorolojisi girdi olarak kullanılacak; yarının gerçekleşen havası kullanılmayacak.

In [ ]:
met_degiskenler = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "surface_pressure", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
]

meteoroloji = open_meteo_cek(
    "https://archive-api.open-meteo.com/v1/archive",
    {
        "latitude": ENLEM, "longitude": BOYLAM, "timezone": SAAT_DILIMI,
        "hourly": ",".join(met_degiskenler), "wind_speed_unit": "ms",
    },
    pd.Timestamp(BASLANGIC) - pd.Timedelta(days=7), BITIS,
)
meteoroloji.columns = ["met_" + c for c in meteoroloji.columns]
print(meteoroloji.shape)
display(meteoroloji.describe().T.round(2))

## 4. Copernicus CAMS hava kalitesi (Open-Meteo Air Quality API)

Önce ~11 km çözünürlüklü **CAMS Avrupa** alanı denenir (Ekim 2023'ten itibaren). Bir kirletici bu alanda yeterince dolu değilse aynı kirletici için **CAMS küresel** (~45 km) alanı yedek olarak çekilir. Hangi kaynağın kullanıldığı tabloda yazılır.

> **Sızıntı notu:** Open-Meteo arşivi, her saat için o güne ait en güncel model çalışmasını saklar. Yani "yarın" için arşivdeki değer, bugün değil yarın yayınlanmış bir tahmindir. Modelde CAMS'i **bugünün CAMS değeri** olarak (ve sınırlılık olarak raporlayarak) kullanacağız.

In [ ]:
cams_degiskenler = {
    "pm10": "PM10", "pm2_5": "PM2_5",
    "nitrogen_dioxide": "NO2", "sulphur_dioxide": "SO2", "dust": "dust",
}
CAMS_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

def cams_cek(alan):
    return open_meteo_cek(
        CAMS_URL,
        {
            "latitude": ENLEM, "longitude": BOYLAM, "timezone": SAAT_DILIMI,
            "hourly": ",".join(cams_degiskenler), "domains": alan,
        },
        pd.Timestamp(BASLANGIC) - pd.Timedelta(days=7), BITIS, parca_gun=90,
    )

cams_avrupa = cams_cek("cams_europe")
doluluk = cams_avrupa.notna().mean().mul(100).round(1)
print("CAMS Avrupa doluluk (%):")
display(doluluk)

eksik_kalanlar = doluluk[doluluk < 80].index.tolist()
cams_kaynak = {d: "cams_europe" for d in cams_degiskenler}

cams = cams_avrupa.copy()
if eksik_kalanlar:
    print("Yedek olarak CAMS küresel çekiliyor:", eksik_kalanlar)
    cams_kuresel = cams_cek("cams_global")
    for d in eksik_kalanlar:
        if cams_kuresel[d].notna().mean() > cams[d].notna().mean():
            # Küresel model 3 saatliktir: aradaki saatleri doğrusal dolduruyoruz (en fazla 2 saat).
            cams[d] = cams_kuresel[d].interpolate(limit=2, limit_area="inside")
            cams_kaynak[d] = "cams_global"

cams = cams.rename(columns=cams_degiskenler).add_prefix("cams_")
display(pd.DataFrame({
    "kaynak": pd.Series(cams_kaynak).rename(index=cams_degiskenler),
    "doluluk (%)": cams.notna().mean().mul(100).round(1).values,
}))

## 5. Saatlik birleştirme

In [ ]:
saatlik = (
    istasyon
    .join(meteoroloji, how="outer")
    .join(cams, how="outer")
)
saatlik.index.name = "tarih"

# Rüzgârı u/v bileşenlerine ayırıyoruz: yön 359° ile 1°'nin yakın olduğunu modelin anlayabilmesi için.
yon_rad = np.deg2rad(saatlik["met_wind_direction_10m"])
saatlik["met_wind_u"] = -saatlik["met_wind_speed_10m"] * np.sin(yon_rad)
saatlik["met_wind_v"] = -saatlik["met_wind_speed_10m"] * np.cos(yon_rad)

print("Saatlik tablo:", saatlik.shape)
display(saatlik.loc[BASLANGIC:].head())

## 6. Günlük özet

- İstasyon: EDA kuralı korunur — en az 18 saat ölçüm varsa **günlük medyan** (model hedefi) ve **günlük ortalama** (WHO eşikleri ortalamaya göre tanımlı).
- CAMS: günlük ortalama ve maksimum (toz olayları için).
- Meteoroloji: sıcaklık/nem/basınç ortalaması, yağış toplamı, rüzgâr hızı ortalaması ve maksimum hamlesi; rüzgâr yönü u/v ortalamasından (vektörel) hesaplanır.

In [ ]:
gun = saatlik.resample("D")
sayim = gun[kirleticiler].count()

gunluk = pd.concat(
    [
        gun[kirleticiler].median().where(sayim >= 18).add_suffix("_medyan"),
        gun[kirleticiler].mean().where(sayim >= 18).add_suffix("_ortalama"),
        sayim.add_suffix("_saat"),
        gun[[c for c in saatlik if c.startswith("cams_")]].mean().add_suffix("_ort"),
        gun[["cams_dust", "cams_PM10"]].max().add_suffix("_maks"),
        gun[["met_temperature_2m", "met_relative_humidity_2m", "met_surface_pressure",
             "met_wind_speed_10m", "met_wind_u", "met_wind_v"]].mean(),
        gun[["met_precipitation"]].sum(min_count=18),
        gun[["met_wind_gusts_10m"]].max(),
    ],
    axis=1,
)

gunluk["met_wind_dir_vektorel"] = (
    np.rad2deg(np.arctan2(-gunluk["met_wind_u"], -gunluk["met_wind_v"])) % 360
)
gunluk.index.name = "tarih"

print("Günlük tablo:", gunluk.shape)
display(gunluk.loc[BASLANGIC:].head())

## 7. Hızlı kontroller

In [ ]:
# 7a) Kapsam: istasyon döneminde her kaynağın doluluk oranı
donem = gunluk.loc[BASLANGIC:BITIS]
kapsam = donem.notna().mean().mul(100).round(1)
display(kapsam.to_frame("dolu gün (%)"))

# 7b) CAMS istasyonu ne kadar tutturuyor? (günlük ortalamalar)
kontrol = []
for k in kirleticiler:
    ortak = donem[[f"{k}_ortalama", f"cams_{k}_ort"]].dropna()
    fark = ortak[f"cams_{k}_ort"] - ortak[f"{k}_ortalama"]
    kontrol.append({
        "Kirletici": k,
        "Gün": len(ortak),
        "İstasyon ort.": ortak[f"{k}_ortalama"].mean().round(1),
        "CAMS ort.": ortak[f"cams_{k}_ort"].mean().round(1),
        "Ortalama sapma (CAMS − istasyon)": fark.mean().round(1),
        "MAE": fark.abs().mean().round(1),
        "Korelasyon (r)": ortak.corr().iloc[0, 1].round(2),
    })
display(pd.DataFrame(kontrol))

In [ ]:
# 7c) Saat dilimi kontrolü: istasyon ile CAMS saatlik NO2 arasındaki korelasyon
# hangi kaydırmada en yüksek? Zirve 0'da çıkarsa iki kaynak aynı saat düzenindedir.
s = saatlik.loc[BASLANGIC:BITIS]
kaydirma = pd.Series(
    {h: s["NO2"].corr(s["cams_NO2"].shift(h)) for h in range(-4, 5)},
    name="r",
)
kaydirma.index.name = "CAMS kaydırma (saat)"
display(kaydirma.round(3).to_frame())
print("En yüksek korelasyon kaydırması:", kaydirma.idxmax(), "saat")

# Ortalama gün içi profil: istasyon vs CAMS
fig, eksen = plt.subplots(1, 2, figsize=(12, 3.5))
for i, k in enumerate(["NO2", "PM10"]):
    profil = s.groupby(s.index.hour)[[k, f"cams_{k}"]].median()
    profil.plot(ax=eksen[i], marker="o", ms=3)
    eksen[i].set_title(f"{k}: saatlik medyan profil")
    eksen[i].set_xlabel("Saat (yerel)")
    eksen[i].set_ylabel("µg/m³")
    eksen[i].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Kaydet ve indir

In [ ]:
import os
os.makedirs("processed", exist_ok=True)

saatlik_dosya = "processed/arnavutkoy_hourly_merged.csv"
gunluk_dosya = "processed/arnavutkoy_daily_merged.csv"

saatlik.loc[: BITIS + " 23:00"].round(3).to_csv(saatlik_dosya)
gunluk.loc[:BITIS].round(3).to_csv(gunluk_dosya)

print("Kaydedildi:", saatlik_dosya, gunluk_dosya)

try:
    from google.colab import files
    files.download(saatlik_dosya)
    files.download(gunluk_dosya)
except ImportError:
    pass